In [1]:
import os
import psycopg2
from pgvector.psycopg2 import register_vector
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv

load_dotenv()
db_params = {
    "dbname": "data-sense-db",
    "user": os.getenv("POSTGRES_USERNAME"),
    "password": os.getenv("POSTGRES_PASSWORD"),
    "host": os.getenv("POSTGRES_HOST"),
    "port": "5432",
}
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")

d:\ai-learning\data-sense\agent-service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
BATCH_SIZE = 32
embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        encode_kwargs={"normalize_embeddings": True, "batch_size": BATCH_SIZE},
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9171.87it/s]


In [3]:
conn = psycopg2.connect(**db_params)
register_vector(conn)
cur = conn.cursor()

In [4]:
RETRIEVAL_SQL = """
    SELECT
        table_name,
        ROUND(
            MAX(
                (1 - (embedding <=> %s::vector)) *
                CASE chunk_type
                    WHEN 'use_cases' THEN 1.4
                    WHEN 'overview'  THEN 1.1
                    ELSE                  1.0
                END
            )::numeric, 4
        ) AS weighted_similarity
    FROM schema_chunks
    GROUP BY table_name
    ORDER BY MAX(
        (1 - (embedding <=> %s::vector)) *
        CASE chunk_type
            WHEN 'use_cases' THEN 1.4
            WHEN 'overview'  THEN 1.1
            ELSE                  1.0
        END
    ) DESC
    LIMIT %s
"""

In [5]:
EASY_TEST_QUESTIONS = [
    "How many products are listed in the Electronics category?",
    "List all sellers registered in Karnataka state.",
    "How many female customers are registered from Gujarat?",
    "What is the average seller rating for verified Brand Store sellers?",
    "How many orders were delivered using BlueDart?"
]

MEDIUM_TEST_QUESTIONS = [
    "What is the total revenue generated by each seller?",
    "Which products are out of stock in the Delhi warehouse?",
    "Show the monthly count of cancelled orders for 2023.",
    "How many orders used the DIWALI30 coupon code and what was the total discount given?",
    "Which categories have the highest average selling price after discount?"
]

HARD_TEST_QUESTIONS = [
    "Which Paytm wallet users bought Electronics products worth more than ₹5000?",
    "Which sellers in Mumbai have stock below reorder level for Fashion products?",
    "What is the average EMI tenure for Samsung products bought on EMI?",
    "How many UPI orders from Uttar Pradesh customers were delivered by Ekart?",
    "Show month-wise revenue trend for the Books category in 2023."
]

TOP_K = 5

In [6]:
def run_retrieval_test(conn, embeddings, questions):
    cur = conn.cursor()
    print("\n" + "=" * 60)
    print(f"  Retrieval Test  (top_k={TOP_K}, weighted by chunk_type)")
    print("=" * 60)

    for question in questions:
        vec = embeddings.embed_query(question)
        cur.execute(RETRIEVAL_SQL, (vec, vec, TOP_K))
        rows = cur.fetchall()

        if not rows:
            print(f"\nQ: {question[:65]}\n   ✗ No results")
            continue

        max_sim = float(rows[0][1])
        flag = "  ⚠ LOW" if max_sim < 0.35 else ""
        result = "  →  ".join(f"{r[0]} ({r[1]})" for r in rows)
        q_short = question[:65] + "…" if len(question) > 65 else question
        print(f"\nQ: {q_short}")
        print(f"   {result}{flag}")

    cur.close()

In [7]:
run_retrieval_test(conn, embeddings, EASY_TEST_QUESTIONS)


  Retrieval Test  (top_k=5, weighted by chunk_type)

Q: How many products are listed in the Electronics category?
   products (0.6868)  →  order_items (0.4344)  →  inventory (0.3897)  →  sellers (0.3584)  →  orders (0.2697)

Q: List all sellers registered in Karnataka state.
   sellers (0.6513)  →  customers (0.5407)  →  inventory (0.4707)  →  order_items (0.4566)  →  products (0.4358)

Q: How many female customers are registered from Gujarat?
   customers (0.7222)  →  sellers (0.4585)  →  orders (0.4313)  →  inventory (0.3470)  →  order_payments (0.2791)

Q: What is the average seller rating for verified Brand Store seller…
   sellers (0.6953)  →  products (0.4800)  →  order_items (0.3742)  →  customers (0.3726)  →  orders (0.2420)

Q: How many orders were delivered using BlueDart?
   orders (0.6441)  →  inventory (0.4812)  →  order_items (0.4480)  →  order_payments (0.4446)  →  sellers (0.3608)


In [8]:
run_retrieval_test(conn, embeddings, MEDIUM_TEST_QUESTIONS)


  Retrieval Test  (top_k=5, weighted by chunk_type)

Q: What is the total revenue generated by each seller?
   order_items (0.5618)  →  sellers (0.5121)  →  products (0.4491)  →  orders (0.3400)  →  customers (0.3273)

Q: Which products are out of stock in the Delhi warehouse?
   inventory (0.7815)  →  orders (0.5946)  →  products (0.4559)  →  sellers (0.4437)  →  order_items (0.4244)

Q: Show the monthly count of cancelled orders for 2023.
   orders (0.5506)  →  inventory (0.5188)  →  order_payments (0.4990)  →  customers (0.4811)  →  order_items (0.4225)

Q: How many orders used the DIWALI30 coupon code and what was the to…
   orders (0.7694)  →  customers (0.4369)  →  order_items (0.4146)  →  order_payments (0.4127)  →  products (0.3699)

Q: Which categories have the highest average selling price after dis…
   products (0.5311)  →  order_items (0.5212)  →  sellers (0.4093)  →  inventory (0.3196)  →  customers (0.3086)


In [9]:
run_retrieval_test(conn, embeddings, HARD_TEST_QUESTIONS)


  Retrieval Test  (top_k=5, weighted by chunk_type)

Q: Which Paytm wallet users bought Electronics products worth more t…
   customers (0.4533)  →  order_payments (0.4455)  →  products (0.3993)  →  orders (0.3915)  →  sellers (0.3497)

Q: Which sellers in Mumbai have stock below reorder level for Fashio…
   inventory (0.7806)  →  sellers (0.5251)  →  products (0.4841)  →  orders (0.4509)  →  order_items (0.4492)

Q: What is the average EMI tenure for Samsung products bought on EMI…
   order_items (0.3470)  →  orders (0.3434)  →  order_payments (0.3351)  →  customers (0.3343)  →  sellers (0.3027)  ⚠ LOW

Q: How many UPI orders from Uttar Pradesh customers were delivered b…
   orders (0.7349)  →  customers (0.5653)  →  inventory (0.5500)  →  order_items (0.4128)  →  order_payments (0.4073)

Q: Show month-wise revenue trend for the Books category in 2023.
   products (0.5038)  →  order_items (0.4340)  →  customers (0.3546)  →  inventory (0.3509)  →  order_payments (0.3476)


In [10]:
conn.close()